# Audio Preprocessing + Segmentation

**Purpose:** Take raw downloaded MP3 audio files (which may be 30–40 minutes long from YouTube), clean them (noise reduction), and **segment them into 10–15 second clips** suitable for ASR training. Clips that are mostly silence are discarded.

**What this notebook does (in order):**
1. Load raw audio and resample to 16kHz mono
2. Apply noise reduction on the full file
3. Detect speech segments using energy-based voice activity detection
4. Cut the audio into 10–15 second clips at silence boundaries
5. Discard clips that are mostly silence (below speech energy threshold)
6. Normalize volume and save each clip as a separate WAV file

**What gets saved:**
- Cleaned WAV clips → `CSE499_EHR_Project/01_Dataset/cleaned_audio/[dialect]/`
- Clip naming: `[dialect]_[filenum]_[gender]_[age]_clip[NNN].wav`
  - Example: `nb_001_male_30s_clip001.wav`, `nb_001_male_30s_clip002.wav`, ...

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/CSE499_EHR_Project'
assert os.path.exists(ROOT), f'ERROR: {ROOT} not found. Run 00_project_setup.ipynb first.'
print(f'✅ Drive mounted. Project root: {ROOT}')

Mounted at /content/drive
✅ Drive mounted. Project root: /content/drive/MyDrive/CSE499_EHR_Project


## Install audio processing libraries


In [ ]:
!pip install -q librosa noisereduce soundfile pydub

# pydub needs ffmpeg for MP3 reading (already installed in Colab)
!which ffmpeg
print('✅ All audio libraries installed')

/usr/bin/ffmpeg
✅ All audio libraries installed


## Define paths and segmentation parameters
Source: raw MP3 files in `raw_audio/[dialect]/` (these may be 30–40 minutes long)  
Destination: cleaned 10–15 second WAV clips in `cleaned_audio/[dialect]/`  


In [ ]:
import os

RAW_AUDIO_DIR = f'{ROOT}/01_Dataset/raw_audio'
CLEANED_AUDIO_DIR = f'{ROOT}/01_Dataset/cleaned_audio'

DIALECT_FOLDERS = ['puran_dhaka', 'barishal', 'sylheti', 'normal_bangla', 'indian_bangla']

# Create dialect subfolders inside cleaned_audio/ if they do not exist
for folder in DIALECT_FOLDERS:
    path = os.path.join(CLEANED_AUDIO_DIR, folder)
    os.makedirs(path, exist_ok=True)

# ---- Segmentation Parameters ----
TARGET_SR = 16000           # 16kHz — standard for ASR models
MIN_CLIP_SECONDS = 10       # Minimum clip duration (seconds)
MAX_CLIP_SECONDS = 15       # Maximum clip duration (seconds)
SILENCE_THRESH_DB = 25      # dB below peak to consider as silence
MIN_SILENCE_MS = 300        # Minimum silence duration (ms) to be a valid split point
MIN_SPEECH_RATIO = 0.3      # Discard clip if speech occupies less than 30% of it

print('✅ Paths and parameters set:')
print(f'   Raw audio:     {RAW_AUDIO_DIR}')
print(f'   Cleaned clips: {CLEANED_AUDIO_DIR}')
print(f'   Clip length:   {MIN_CLIP_SECONDS}–{MAX_CLIP_SECONDS} seconds')
print(f'   Silence threshold: {SILENCE_THRESH_DB} dB')
print(f'   Min speech ratio:  {MIN_SPEECH_RATIO} (discard if less than {int(MIN_SPEECH_RATIO*100)}% speech)')
print()
for folder in DIALECT_FOLDERS:
    print(f'   ✅ {CLEANED_AUDIO_DIR}/{folder}/')

✅ Paths and parameters set:
   Raw audio:     /content/drive/MyDrive/CSE499_EHR_Project/01_Dataset/raw_audio
   Cleaned clips: /content/drive/MyDrive/CSE499_EHR_Project/01_Dataset/cleaned_audio
   Clip length:   10–15 seconds
   Silence threshold: 25 dB
   Min speech ratio:  0.3 (discard if less than 30% speech)

   ✅ /content/drive/MyDrive/CSE499_EHR_Project/01_Dataset/cleaned_audio/puran_dhaka/
   ✅ /content/drive/MyDrive/CSE499_EHR_Project/01_Dataset/cleaned_audio/barishal/
   ✅ /content/drive/MyDrive/CSE499_EHR_Project/01_Dataset/cleaned_audio/sylheti/
   ✅ /content/drive/MyDrive/CSE499_EHR_Project/01_Dataset/cleaned_audio/normal_bangla/
   ✅ /content/drive/MyDrive/CSE499_EHR_Project/01_Dataset/cleaned_audio/indian_bangla/


## Define the audio cleaning + segmentation pipeline

1. **`find_split_points()`** — Analyzes the audio to find silence gaps where we can safely cut without splitting a word in half. Uses `librosa.effects.split()` to detect non-silent intervals, then identifies gaps between them.

2. **`segment_audio()`** — Takes the full cleaned audio array and splits it into clips of 10–15 seconds. It tries to cut at silence boundaries. If no silence boundary exists within the target range, it forces a cut at exactly MAX_CLIP_SECONDS. Each clip is checked: if it is mostly silence (speech < 30% of duration), it is discarded.

3. **`process_and_segment_file()`** — The main function that handles one raw audio file end-to-end: load → noise reduce → segment → normalize → save each clip to Drive.

In [ ]:
import librosa
import noisereduce as nr
import soundfile as sf
import numpy as np


def find_split_points(audio, sr, top_db=SILENCE_THRESH_DB, min_silence_samples=None):

    if min_silence_samples is None:
        min_silence_samples = int(MIN_SILENCE_MS / 1000 * sr)

    # Get intervals where audio is NOT silent
    # Each row is [start_sample, end_sample] of a non-silent segment
    non_silent = librosa.effects.split(audio, top_db=top_db)

    if len(non_silent) < 2:
        # No silence gaps found — audio is continuous speech or continuous silence
        return []

    split_points = []
    for i in range(len(non_silent) - 1):
        gap_start = non_silent[i][1]       # End of current speech segment
        gap_end = non_silent[i + 1][0]     # Start of next speech segment
        gap_length = gap_end - gap_start

        if gap_length >= min_silence_samples:
            # Use the midpoint of the silence gap as the split point
            midpoint = gap_start + gap_length // 2
            split_points.append(midpoint)

    return sorted(split_points)


def compute_speech_ratio(clip, sr, top_db=SILENCE_THRESH_DB):


    non_silent = librosa.effects.split(clip, top_db=top_db)
    if len(non_silent) == 0:
        return 0.0

    speech_samples = sum(end - start for start, end in non_silent)
    return speech_samples / len(clip)


def segment_audio(audio, sr, min_sec=MIN_CLIP_SECONDS, max_sec=MAX_CLIP_SECONDS,
                  min_speech_ratio=MIN_SPEECH_RATIO):

    total_samples = len(audio)
    min_samples = int(min_sec * sr)
    max_samples = int(max_sec * sr)

    # If the audio is already short enough, return it as a single clip
    total_duration = total_samples / sr
    if total_duration <= max_sec:
        ratio = compute_speech_ratio(audio, sr)
        if ratio >= min_speech_ratio and total_duration >= 1.0:
            return [(audio, 0.0, total_duration, ratio)]
        else:
            return []

    # Find all candidate split points (silence midpoints)
    split_points = find_split_points(audio, sr)

    clips = []
    current_start = 0  # Current position in samples

    while current_start < total_samples:
        remaining = total_samples - current_start

        # If what's left is shorter than min_sec, take it all (if long enough to matter)
        if remaining <= max_samples:
            clip = audio[current_start:]
            clip_duration = len(clip) / sr
            if clip_duration >= 3.0:  # Only keep tail clips at least 3 seconds
                ratio = compute_speech_ratio(clip, sr)
                if ratio >= min_speech_ratio:
                    clips.append((clip, current_start / sr, total_samples / sr, ratio))
            break

        # Look for the best split point between min_samples and max_samples from current_start
        window_start = current_start + min_samples
        window_end = current_start + max_samples

        # Find split points that fall within our target window
        candidates = [sp for sp in split_points if window_start <= sp <= window_end]

        if candidates:
            # Pick the split point closest to the middle of the window (best balance)
            window_mid = (window_start + window_end) // 2
            best_split = min(candidates, key=lambda sp: abs(sp - window_mid))
            cut_point = best_split
        else:
            # No silence gap in the window — force-cut at max_sec
            cut_point = current_start + max_samples

        # Extract the clip
        clip = audio[current_start:cut_point]
        clip_duration = len(clip) / sr

        # Check speech ratio — discard if mostly silence
        ratio = compute_speech_ratio(clip, sr)
        if ratio >= min_speech_ratio:
            clips.append((clip, current_start / sr, cut_point / sr, ratio))

        # Move forward
        current_start = cut_point

    return clips


def process_and_segment_file(input_path, output_dir, base_name, target_sr=TARGET_SR):

    try:
        # Step 1: Load full audio
        audio, sr = librosa.load(input_path, sr=target_sr, mono=True)

        if len(audio) == 0:
            return 0, 0, 0.0, 'Empty audio file'

        full_duration = len(audio) / sr

        # Step 2: Noise reduction on the full file
        # Processing the whole file at once gives noisereduce a better noise profile
        audio_cleaned = nr.reduce_noise(
            y=audio,
            sr=sr,
            prop_decrease=0.7,
            stationary=True
        )

        # Step 3: Segment into 10–15 second clips
        clips = segment_audio(audio_cleaned, sr)

        # Step 4: Save each clip
        saved = 0
        discarded = 0
        total_saved_duration = 0.0

        for idx, (clip_audio, start_sec, end_sec, speech_ratio) in enumerate(clips):
            clip_num = str(idx + 1).zfill(3)
            clip_filename = f'{base_name}_clip{clip_num}.wav'
            clip_path = os.path.join(output_dir, clip_filename)

            # Skip if this clip already exists (resume support)
            if os.path.exists(clip_path):
                saved += 1
                total_saved_duration += len(clip_audio) / sr
                continue

            # Normalize volume
            max_val = np.max(np.abs(clip_audio))
            if max_val > 0:
                clip_audio = clip_audio / max_val * 0.95

            # Save to Drive
            sf.write(clip_path, clip_audio, sr)
            saved += 1
            total_saved_duration += len(clip_audio) / sr

        # Total clips that were generated but discarded due to low speech ratio
        # (this is already handled inside segment_audio — clips below threshold
        #  are never returned. But we can estimate from the gap.)
        expected_clips = int(full_duration / ((MIN_CLIP_SECONDS + MAX_CLIP_SECONDS) / 2))
        discarded = max(0, expected_clips - len(clips))

        return saved, discarded, total_saved_duration, None

    except Exception as e:
        return 0, 0, 0.0, str(e)


print('✅ Functions defined:')
print('   find_split_points()       — finds silence gaps for safe cutting')
print('   compute_speech_ratio()    — checks if a clip has enough speech')
print('   segment_audio()           — splits long audio into 10–15s clips')
print('   process_and_segment_file() — full pipeline for one raw file')

✅ Functions defined:
   find_split_points()       — finds silence gaps for safe cutting
   compute_speech_ratio()    — checks if a clip has enough speech
   segment_audio()           — splits long audio into 10–15s clips
   process_and_segment_file() — full pipeline for one raw file


## Process and segment all raw audio files
finds all raw audio files (MP3), runs the full clean+segment pipeline, and saves the resulting clips to `cleaned_audio/[dialect]/`.

**Clip naming convention:** `[original_name_without_ext]_clip[NNN].wav`  
Example: raw file `nb_001_male_30s.mp3` → clips `nb_001_male_30s_clip001.wav`, `nb_001_male_30s_clip002.wav`, ...

In [ ]:
from tqdm import tqdm

total_files_processed = 0
total_files_skipped = 0
total_files_failed = 0
total_clips_created = 0
total_clips_discarded = 0
total_duration_saved = 0.0

for dialect_folder in DIALECT_FOLDERS:
    raw_dir = os.path.join(RAW_AUDIO_DIR, dialect_folder)
    clean_dir = os.path.join(CLEANED_AUDIO_DIR, dialect_folder)

    if not os.path.exists(raw_dir):
        print(f'⚠️  {dialect_folder}: raw_audio folder not found — skipping')
        continue

    # Get all audio files in this dialect folder
    audio_files = sorted([
        f for f in os.listdir(raw_dir)
        if f.endswith(('.mp3', '.wav', '.m4a', '.flac', '.ogg'))
    ])

    if len(audio_files) == 0:
        print(f'⚠️  {dialect_folder}: No audio files found')
        continue

    print(f'\n🔧 Processing {dialect_folder}/ ({len(audio_files)} raw files)')

    for filename in tqdm(audio_files, desc=dialect_folder):
        base_name = os.path.splitext(filename)[0]
        input_path = os.path.join(raw_dir, filename)

        # ---- Resume check: skip if .done marker exists ----
        done_marker = os.path.join(clean_dir, f'.{base_name}.done')
        if os.path.exists(done_marker):
            total_files_skipped += 1
            continue

        # ---- Run the full pipeline ----
        num_saved, num_discarded, dur_saved, error = process_and_segment_file(
            input_path=input_path,
            output_dir=clean_dir,
            base_name=base_name
        )

        if error:
            total_files_failed += 1
            print(f'   ❌ Failed: {filename} — {error}')
        else:
            total_files_processed += 1
            total_clips_created += num_saved
            total_clips_discarded += num_discarded
            total_duration_saved += dur_saved

            # Write .done marker so this file is skipped on re-run
            with open(done_marker, 'w') as f:
                f.write(f'clips:{num_saved} discarded:{num_discarded} duration:{dur_saved:.1f}s')

            tqdm.write(
                f'   ✅ {filename} → {num_saved} clips '
                f'({dur_saved:.0f}s saved, ~{num_discarded} silence clips discarded)'
            )

print(f'\n{"=" * 55}')
print(f'  Audio Preprocessing + Segmentation Complete')
print(f'{"=" * 55}')
print(f'  Raw files processed: {total_files_processed}')
print(f'  Raw files skipped (already done): {total_files_skipped}')
print(f'  Raw files failed: {total_files_failed}')
print(f'  ---')
print(f'  Total clips created: {total_clips_created}')
print(f'  Total clips discarded (silence): ~{total_clips_discarded}')
print(f'  Total audio saved: {total_duration_saved:.0f}s ({total_duration_saved/60:.1f} min)')


🔧 Processing puran_dhaka/ (1 raw files)


puran_dhaka: 100%|██████████| 1/1 [01:01<00:00, 61.56s/it]


   ✅ pd_001_male_40s.mp3 → 153 clips (1966s saved, ~18 silence clips discarded)

🔧 Processing barishal/ (2 raw files)


barishal:  50%|█████     | 1/2 [00:23<00:23, 23.55s/it]

   ✅ br_001_male_25s.mp3 → 98 clips (1402s saved, ~14 silence clips discarded)


...

   ✅ nb_003_female_40s.mp3 → 360 clips (4668s saved, ~13 silence clips discarded)

🔧 Processing indian_bangla/ (2 raw files)


indian_bangla:  50%|█████     | 1/2 [00:59<00:59, 59.51s/it]

   ✅ ib_001_male_50s.mp3 → 296 clips (3786s saved, ~6 silence clips discarded)


indian_bangla: 100%|██████████| 2/2 [01:29<00:00, 44.74s/it]

   ✅ ib_002_male_50s.mp3 → 156 clips (1965s saved, ~1 silence clips discarded)

  Audio Preprocessing + Segmentation Complete
  Raw files processed: 10
  Raw files skipped (already done): 0
  Raw files failed: 0
  ---
  Total clips created: 1288
  Total clips discarded (silence): ~73
  Total audio saved: 16751s (279.2 min)


## Verify cleaned audio clips


In [ ]:
print('=== Cleaned audio clips per dialect ===')
grand_total = 0
grand_duration = 0.0

for folder in DIALECT_FOLDERS:
    path = os.path.join(CLEANED_AUDIO_DIR, folder)
    if not os.path.exists(path):
        print(f'  {folder}: folder not found')
        continue

    clips = sorted([f for f in os.listdir(path) if f.endswith('.wav')])
    grand_total += len(clips)

    # Calculate total duration of clips
    folder_duration = 0.0
    for clip_file in clips:
        clip_path = os.path.join(path, clip_file)
        try:
            info = sf.info(clip_path)
            folder_duration += info.duration
        except:
            pass

    grand_duration += folder_duration
    print(f'  {folder}: {len(clips)} clips ({folder_duration:.0f}s = {folder_duration/60:.1f} min)')

    # Show clip duration range
    if clips:
        durations = []
        for clip_file in clips[:20]:  # Sample first 20 for speed
            try:
                info = sf.info(os.path.join(path, clip_file))
                durations.append(info.duration)
            except:
                pass
        if durations:
            print(f'           Clip durations (sample): {min(durations):.1f}s – {max(durations):.1f}s (avg {np.mean(durations):.1f}s)')

print(f'\n  TOTAL: {grand_total} clips ({grand_duration:.0f}s = {grand_duration/60:.1f} min)')

=== Cleaned audio clips per dialect ===
  puran_dhaka: 153 clips (1966s = 32.8 min)
           Clip durations (sample): 10.6s – 15.0s (avg 12.8s)
  barishal: 144 clips (2001s = 33.4 min)
           Clip durations (sample): 10.4s – 15.0s (avg 14.5s)
  sylheti: 135 clips (1814s = 30.2 min)
           Clip durations (sample): 10.2s – 15.0s (avg 12.5s)
  normal_bangla: 404 clips (5219s = 87.0 min)
           Clip durations (sample): 11.1s – 14.5s (avg 12.4s)
  indian_bangla: 452 clips (5751s = 95.9 min)
           Clip durations (sample): 10.8s – 15.0s (avg 13.2s)

  TOTAL: 1288 clips (16751s = 279.2 min)


## Play sample clips


In [ ]:
import IPython.display as ipd

# Change these to listen to different clips
sample_dialect = 'normal_bangla'
sample_index = 0  # 0 = first clip
num_samples = 3   # How many consecutive clips to play

sample_dir = os.path.join(CLEANED_AUDIO_DIR, sample_dialect)
wav_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.wav')])

if len(wav_files) == 0:
    print(f'⚠️  No clips found in {sample_dialect}/. Run the processing cell first.')
else:
    for i in range(sample_index, min(sample_index + num_samples, len(wav_files))):
        sample_path = os.path.join(sample_dir, wav_files[i])
        audio_data, sr = librosa.load(sample_path, sr=16000)
        duration = len(audio_data) / sr
        print(f'\n🔊 Clip {i+1}: {wav_files[i]} ({duration:.1f}s)')
        ipd.display(ipd.Audio(audio_data, rate=sr))


🔊 Clip 1: nb_001_male_30s_clip001.wav (14.5s)



🔊 Clip 2: nb_001_male_30s_clip002.wav (11.1s)



🔊 Clip 3: nb_001_male_30s_clip003.wav (12.6s)


## List all clip filenames for reference
Shows every clip filename.


In [ ]:
print('=== All clip filenames by dialect ===')
for folder in DIALECT_FOLDERS:
    path = os.path.join(CLEANED_AUDIO_DIR, folder)
    if not os.path.exists(path):
        continue

    clips = sorted([f for f in os.listdir(path) if f.endswith('.wav')])
    if not clips:
        continue

    print(f'\n📁 {folder}/ ({len(clips)} clips)')

    # Group clips by source file
    from collections import defaultdict
    groups = defaultdict(list)
    for clip in clips:
        # Source file name = everything before _clipNNN
        if '_clip' in clip:
            source = clip.rsplit('_clip', 1)[0]
        else:
            source = clip  # Not a segmented clip (shouldn't happen normally)
        groups[source].append(clip)

    for source, source_clips in sorted(groups.items()):
        print(f'  {source} → {len(source_clips)} clips')
        for c in source_clips[:5]:
            print(f'    {c}')
        if len(source_clips) > 5:
            print(f'    ... and {len(source_clips) - 5} more')

=== All clip filenames by dialect ===

📁 puran_dhaka/ (153 clips)
  pd_001_male_40s → 153 clips
    pd_001_male_40s_clip001.wav
    pd_001_male_40s_clip002.wav
    pd_001_male_40s_clip003.wav
    pd_001_male_40s_clip004.wav
    pd_001_male_40s_clip005.wav
    ... and 148 more

📁 barishal/ (144 clips)
  br_001_male_25s → 98 clips
    br_001_male_25s_clip001.wav
    br_001_male_25s_clip002.wav
    br_001_male_25s_clip003.wav
    br_001_male_25s_clip004.wav
    br_001_male_25s_clip005.wav
    ... and 93 more
  br_002_female_30s → 46 clips
    br_002_female_30s_clip001.wav
    br_002_female_30s_clip002.wav
    br_002_female_30s_clip003.wav
    br_002_female_30s_clip004.wav
    br_002_female_30s_clip005.wav
    ... and 41 more

📁 sylheti/ (135 clips)
  sy_001_male_26s → 59 clips
    sy_001_male_26s_clip001.wav
    sy_001_male_26s_clip002.wav
    sy_001_male_26s_clip003.wav
    sy_001_male_26s_clip004.wav
    sy_001_male_26s_clip005.wav
    ... and 54 more
  sy_002_male_26s → 76 clips
    sy

---
## ✅ Complete

**What was saved to Google Drive:**
- Cleaned 10–15 second WAV clips in `01_Dataset/cleaned_audio/[dialect]/`
- Each clip: 16kHz, mono, noise-reduced, volume-normalized
- Clips that were mostly silence have been discarded
- `.done` marker files track which raw files have been fully processed

**Clip naming:** `[dialect]_[filenum]_[gender]_[age]_clip[NNN].wav`